# Practical Medical Record Pseudonymization with the Wowool SDK

This notebook walks through building a clinical text pseudonymization pipeline using the Wowool SDK. 

### Objectives:
1. Run a baseline out-of-the-box anonymization pass.
2. Define custom `.wow` pattern rules for medical record numbers, policy IDs, and age generalization.
3. Configure Python f-string formatters for consistent graph identifiers (`Patient_1`) and dynamic masking.
4. While we strip direct identifiers, we want to retain useful demographic indicators like gender and age for population-level studies
5. Audit the sanitized corpus for residual risk using the `UnknownThing` entity.


We will implement this pipeline using a combination of Wowool’s out-of-the-box entities and custom domain rules.

# Setup and Package Installation

In [ ]:
# Install the required Wowool SDK packages
!pip install nlp-wowool-sdk wowool-anonymizer wowool-english

### Set up your API key

Sign in to wowool.com to generate an API key.

For local development, create a .env file in your project directory:

WOWOOL_SDK_KEY="your-api-key-here"

Install dotenv to read the environment variables:

In [ ]:
!pip install python-dotenv

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()


Create Sample Medical Records

In [ ]:
from pathlib import Path

# Create directories for data, rules, and output
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "sample_record.txt"

file_path.write_text("""# MEDICAL REPORT - CONFIDENTIAL

Patient ID: MR-2024-789456
Date: March 15, 2024
Hospital: St. Mary's General Hospital
Department: Internal Medicine

## PATIENT INFORMATION:
Name: Sarah Johnson
DOB: 08/12/1978 (Age: 45)
Gender: Female
Address: 1247 Oak Street, Springfield, IL 62701
Phone: (217) 555-0198
Insurance: Blue Cross Blue Shield - Policy #: BC887456123

## ATTENDING PHYSICIAN:
Dr. Michael Chen, MD
Internal Medicine
License #: IL-MD-456789

## CLINICAL FINDINGS:
- Potassium: 4.2 mmol/L
- BP: 120/80 mmHg
- Assessment: Mild hypercholesterolemia, chronic obstructive pulmonary disease.
"""
)


print("Sample record written to", file_path)

print("Working Directory:", Path.cwd())
print("Full Target Path: ", file_path.absolute())

## Baseline Anonymization



In [ ]:
from wowool.sdk import Pipeline

# run the anonymizer domain that detects the entities to need to be anonymized.
pipeline = Pipeline(
    [
        "english",
        "anonymizer",  # domain that detects the entities to need to be anonymized
        # Let's run the anonymizer app with the list of entities to anonymize
        {
            "name": "anonymizer.app",
            "options": {
                # which entities to anonymize
                "annotations": [ "Person", "Address", "Date", "PhoneNr" ]
            }
        }
    ]
)


text = Path("data/sample_record.txt").read_text()

# Process the input text using the created pipeline
# The anonymizer domain will detect the entities to be anonymized and the anonymizer app will replace them with anonymized values.
doc = pipeline(text)

# sanity check to ensure that the anonymizer domain ran successfully and produced results
assert doc.anonymizer is not None, "Anonymizer domain did not run successfully."

print(doc.anonymizer.text)

print("-" * 80)

# Show the anonymized entities and their locations in the text in a comprenhensible format
for location in doc.anonymizer.locations:
    print(
        f"({location.begin_offset:>4},{location.end_offset:>4}) {location.uri:<15}: {location.text:<42} -> {location.anonymized}"
    )



This baseline leaves several gaps:

* Custom identifiers like Patient ID, Policy #, and License # were missed because they require domain-specific patterns.

* The anonymizer used a synthetic name by default for Person, Company and Organization. For our graph database, we need a consistent identifier (e.g., #Patient_1).

* Exact age values: for our purposes, we want to classify the patients in age groups, so that the identity of the patient cannot be deduced, but we keep relevant data for further analysis.


We can solve these issues by creating custom rules and customizing our formatters.

## Creating custom rules

To capture domain-specific patterns, create a directory called rules/ and add a .wow rules file (e.g., rules/healthrecord_anonymizer.wow):

> **Note**: if you downloaded the tutorials, you can find this file in your rules directory

> **Note**: the following cell cannot be run, it is a text file

In [ ]:
//---------------------------------------
// PatientID
// Patient IDs start with MR-
//---------------------------------------
rule: { "Patient ID" ":" {"MR-(.)*"}=PatientID };

//---------------------------------------
// Patient
// Create a new annotation just for patients
// Context: Name: John Doe
//---------------------------------------
rule: {"Name" ":" {Person}=Patient };

//---------------------------------------
// PolicyNumber
// Context: Policy #: DGASAS0090888
//---------------------------------------
rule: {"Policy" "#" ":" { <> }=PolicyNumber };

//---------------------------------------
// LicenseNumber
// Context: License #: PA-MD-123456
//---------------------------------------
rule: {"License" "#" ":" { (<>)+ }=LicenseNumber };

//---------------------------------------
// Age Categorization Rules
// 1–12            child
// 13–17           adolescent
// 18–24           young adult
// 25–44           adult
// 45–64           middle-aged adult
// 65+             older adult
//---------------------------------------

// Child (1 to 12 years old)
rule: { 
    Age[ 
        ("[:range(1-9):]" | "1[:range(0-2):]") 
    ] 
} = AgeGroup@(type="child");

// Adolescent (13 to 17 years old)
rule: {
    Age[ "1([:range(3-7):])" ] 
} = AgeGroup@(type="adolescent");

// Young Adult (18 to 24 years old)
rule: { 
    Age[ 
        ( "1(8|9)" | "2[:range(0-4):]" ) 
    ] 
} = AgeGroup@(type="young adult");

// Adult (25 to 44 years old)
rule: { 
    Age[ 
        ( "2[:range(5-9):]" | "3[:digit:]" | "4[:range(0-4):]" ) 
    ] 
} = AgeGroup@(type="adult");

// Middle-Aged Adult (45 to 64 years old)
rule: { 
    Age[ 
        ( "4[:range(5-9):]" | "5[:digit:]" | "6[:range(0-4):]" ) 
    ] 
} = AgeGroup@(type="middle-aged adult");

// Older Adult (65+ years old)
rule: { 
    Age[ 
        (
             "6[:range(5-9):]" 
             | "7[:digit:]" 
             | "8[:digit:]" 
             | "9[:digit:]" 
             | "10[:digit:]"
        ) 
    ] 
} = AgeGroup@(type="older adult");



**Custom rules** let us define domain-specific annotations (PatientID, AgeGroup, etc.) not detected by baseline models. Where possible, make rules as specific as the corpus allows to minimize false matches on unrelated text.

* Append the **rules** directory to the pipeline flag so Wowool evaluates the custom rules alongside the default annotations. 

* Add the newly created **annotations** to the annotations option (AgeGroup,PatientID,Patient,PolicyNumber,LicenseNumber).

In [ ]:
from wowool.document.analysis.document import AnalysisDocument
from wowool.sdk import Pipeline


# run the anonymizer domain that detects the entities to need to be anonymized.
pipeline = Pipeline(
    [
        "english",
        "anonymizer",  # domain that detects the entities to need to be anonymized
        # Let's run the anonymizer app with the list of entities to anonymize
        "rules",       ## Add the folder name where you are storing your custom rules 
        {
            "name": "anonymizer.app",
            "options": {
                # which entities to anonymize
                "annotations": [ "Person", "Address", "Date", "PhoneNr","PatientID","LicenseID","PolicyNumber","LicenseNumber","Patient", "AgeGroup" ]
            }
        }
    ]
)


text = Path("data/sample_record.txt").read_text()

doc: AnalysisDocument = pipeline(text)

assert doc.anonymizer is not None, "Anonymizer domain did not run successfully."
print(doc.anonymizer.text)
print("-" * 80)
for location in doc.anonymizer.locations:
    print(
        f"({location.begin_offset:>4},{location.end_offset:>4}) {location.uri:<15}: {location.text:<42} -> {location.anonymized}"
    )



A few issues remain to be handled in the formatting layer:

* Replace fake doctor names with a generic label like Person_1.

* Mask unnecessary identifiers (e.g., license and policy numbers) with *** instead of semantic labels.

* Map exact age to the AgeGroup attribute created in our rule.

## Customizing the Formatters

Formatters define how matched entities appear in the output. Under the hood, they are evaluated as Python f-strings that expose Wowool objects, attributes, and standard Python string operations. 

Rather than accepting the defaults, we can specify programmatic structural replacements:

* Labels: Entity labels with counters: #Patient_{nr}.

* Entity Attributes & Python Methods: Access metadata directly and transform it on the fly (e.g., {concept.type.replace(" ", "_").upper()} to output MIDDLE_AGED_ADULT).

* Dynamic Masking: Leverage Python expressions like len(literal) to build variable-length masks (#{"*" * len(literal)}#), or assign static replacements (#********#). In our example, we will use static masks, to further anonymize the data, as the length of the string might be an indicator.

> **Note**: We add a # symbol at the begining of the anonymized entities to make it easier to spot them, 

> **Note** In this last version, we are using the the Document method from wowool, to go through the all the files.

In [ ]:
from wowool.document.analysis.document import AnalysisDocument
from wowool.sdk import Pipeline
from wowool.document import Document
from pathlib import Path



# run the anonymizer domain that detects the entities to need to be anonymized.
pipeline = Pipeline(
    [
        "english",
        "anonymizer",  # domain that detects the entities to need to be anonymized
        # Let's run the anonymizer app with the list of entities to anonymize
        "rules",       ## Add the folder name where you are storing your custom rules 
        {
            "name": "anonymizer.app",
            "options": {
                # which entities to anonymize
                "annotations": [ "Person", "Address", "Date", "PhoneNr","PatientID","LicenseID","PolicyNumber","LicenseNumber","Patient", "AgeGroup" ],
                "formatters": {
                    # For patients, add a label and a counter (nr is an predefined variables that increments for each patient found in the text)
                    "Patient": '#Patient_{nr}',
                    # For age groups, we will use the concept type to create a label, replacing spaces with underscores and converting to uppercase
                    "AgeGroup": '#{concept.type.replace(" ", "_").upper()}',
                    # For numerical identifiers, we will use a fixed mask.
                    "PatientID": '#**********',
                    "LicenseID": '#**********',
                    "PolicyNumber": '#**********',
                    "LicenseNumber": '#{literal[:3]}*****',
                }
            }
        }
    ]
)

input_path = Path("data")
output_path = Path("output")
Path(output_path).mkdir(parents=True, exist_ok=True)

for ip in Document.glob(input_path, "*.txt"):
    doc: AnalysisDocument = pipeline(ip)

    assert doc.anonymizer is not None, "Anonymizer domain did not run successfully."
    output_file = Path(output_path) / Path(ip.id).name
    print("out", output_file)

    with open(output_file,"w") as f_out:
        print("Anonymized Doc:", doc.anonymizer.text )
        f_out.write(doc.anonymizer.text)

    # Show the replacements
    for location in doc.anonymizer.locations:
        print(
            f"({location.begin_offset:>4},{location.end_offset:>4}) {location.uri:<15}: {location.text:<42} -> {location.anonymized}"
        )
